In [1]:
import sys
import os
import time
import pandas as pd
import glob

PROJECT_ROOT = r"C:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard"
sys.path.insert(0, PROJECT_ROOT)

from nba_api.stats.endpoints import shotchartdetail

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard


In [2]:
raw_dir = os.path.join(PROJECT_ROOT, "data", "raw")
pattern = os.path.join(raw_dir, "season_stats_*.csv")

all_files = sorted(glob.glob(pattern))

season_files = []

for file in all_files:
    season_str = os.path.basename(file).replace("season_stats_", "").replace(".csv", "")
    year_start = int(season_str.split("-")[0])

    if year_start >= 2010:
        season_files.append(file)

print("Found modern seasons:", len(season_files))
season_files

Found modern seasons: 16


['C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2010-11.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2011-12.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2012-13.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2013-14.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2014-15.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2015-16.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2016-17.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2017-18.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2018-19.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba

In [3]:
def download_shots(player_id, season, season_type):
    sc = shotchartdetail.ShotChartDetail(
        team_id=0,
        player_id=player_id,
        season_nullable=season,
        season_type_all_star=season_type,
        context_measure_simple="FGA"
    )
    return sc.get_data_frames()[0]

In [4]:
shot_data_dir = os.path.join(PROJECT_ROOT, "data", "shot_data")
regular_dir = os.path.join(shot_data_dir, "regular")
playoffs_dir = os.path.join(shot_data_dir, "playoffs")

os.makedirs(regular_dir, exist_ok=True)
os.makedirs(playoffs_dir, exist_ok=True)

In [5]:
for file in season_files:
    season = os.path.basename(file).replace("season_stats_", "").replace(".csv", "")
    print(f"\n=== Processing season {season} ===")

    df_stats = pd.read_csv(file)

    if "PLAYER_ID" not in df_stats.columns:
        raise ValueError(f"PLAYER_ID column missing in {file}")

    players = df_stats[["PLAYER_ID", "PLAYER_NAME"]].drop_duplicates()

    # DataFrames to accumulate all shots
    all_regular = []
    all_playoffs = []

    for _, row in players.iterrows():
        pid = row["PLAYER_ID"]
        name = row["PLAYER_NAME"]

        # Regular Season
        try:
            df_reg = download_shots(pid, season, "Regular Season")
            df_reg["PLAYER_ID"] = pid
            df_reg["PLAYER_NAME"] = name
            all_regular.append(df_reg)

            print(f"Success: Downloaded {name}'s {season} Regular Season data")
            time.sleep(1.2)

        except Exception as e:
            print(f"Failed: {name} {season} Regular Season → {e}")
            time.sleep(3)

        # Playoffs
        try:
            df_po = download_shots(pid, season, "Playoffs")
            df_po["PLAYER_ID"] = pid
            df_po["PLAYER_NAME"] = name
            all_playoffs.append(df_po)

            print(f"Success: Downloaded {name}'s {season} Playoffs data")
            time.sleep(1.2)

        except Exception as e:
            print(f"Failed: {name} {season} Playoffs → {e}")
            time.sleep(3)

    # Save combined files
    if all_regular:
        out_reg = pd.concat(all_regular, ignore_index=True)
        out_reg.to_csv(os.path.join(regular_dir, f"shots_{season}.csv"), index=False)

    if all_playoffs:
        out_po = pd.concat(all_playoffs, ignore_index=True)
        out_po.to_csv(os.path.join(playoffs_dir, f"shots_{season}.csv"), index=False)

    print(f"=== Finished season {season} ===")


=== Processing season 2010-11 ===
Success: Downloaded AJ Price's 2010-11 Regular Season data
Success: Downloaded AJ Price's 2010-11 Playoffs data
Success: Downloaded Aaron Brooks's 2010-11 Regular Season data
Success: Downloaded Aaron Brooks's 2010-11 Playoffs data
Success: Downloaded Aaron Gray's 2010-11 Regular Season data
Success: Downloaded Aaron Gray's 2010-11 Playoffs data
Success: Downloaded Acie Law's 2010-11 Regular Season data
Success: Downloaded Acie Law's 2010-11 Playoffs data
Success: Downloaded Al Harrington's 2010-11 Regular Season data
Success: Downloaded Al Harrington's 2010-11 Playoffs data
Success: Downloaded Al Horford's 2010-11 Regular Season data
Success: Downloaded Al Horford's 2010-11 Playoffs data
Success: Downloaded Al Jefferson's 2010-11 Regular Season data
Success: Downloaded Al Jefferson's 2010-11 Playoffs data
Success: Downloaded Al Thornton's 2010-11 Regular Season data
Success: Downloaded Al Thornton's 2010-11 Playoffs data
Success: Downloaded Al-Farouq

KeyboardInterrupt: 